# Notebook 1 — Segment images with Cellpose 4.0.6 and save masks

## Installation 

### for Mac
Latest stable release is Cellpose 4.0.6, [available via conda-forge](https://anaconda.org/conda-forge/cellpose)

```bash
conda env create -f ./envs/cellpose.yml
conda activate cellpose
```

### for colab

`%pip install "cellpose==4.0.6" "torch" "torchvision" "torchaudio" "scikit-image>=0.22.0" "tqdm>=4.66.0" "pandas>=2.2.0"`

In [1]:
# %% [markdown]
# # Notebook 1 — Cellpose 4.0.6 (cpsam) on Apple Silicon: robust masks for yellow channel
#
# This notebook:
# 1) Loads cpsam (Cellpose-SAM) with PyTorch MPS (Apple Silicon).
# 2) Light per-image intensity rescue (rescale + optional CLAHE + optional denoise).
# 3) Parameter sweep on a few images (cellprob / flow / diameter) to find a sweet spot.
# 4) Segments all yellow images → uint16 masks + RGB overlays + per-image stats CSV.
#
# v4-accurate API notes:
# - Use `cellpose.models.CellposeModel`, not `Cellpose`.
# - Pass `channel_axis` (RGB → -1). Do NOT use legacy `channels=[0,0]`.
# - `SizeModel` is removed in v4; use a fixed `diameter` or try a small grid.
# - Avoid deprecated `rescale` arg; scale is controlled by `diameter`.
# - `normalize` can be bool/dict; we keep False and do our own pre-processing.

# %% [markdown]
# ## 0) Environment quick check

# %%
import sys, platform, torch, os
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # safe fallback for rare ops

# %% [markdown]
# ## 1) Imports and device

# %%
from pathlib import Path
from typing import List
import numpy as np
import pandas as pd
from skimage import io, exposure, filters, measure, morphology, color, util
from tqdm import tqdm
import torch
from cellpose import models

def mps_device():
    return torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else torch.device("cpu")

device = mps_device()
print("Using device:", device)

# %% [markdown]
# ## 2) Project paths & defaults
# *We segment the **yellow** folder only and write results to `analysis/cellpose_results2/yellow/png/`.*

# %%
# Your project
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")
img_root     = project_root / "data"
channels: List[str] = ["yellow"]          # reference masks from yellow
out_root     = project_root / "analysis" / "cellpose_results2"

# File types
image_exts = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]

# Model
pretrained_model = "cpsam"                # stick to cpsam as requested

# Pre-processing (safe defaults for faint IF)
do_rescale_intensity = True               # stretch each channel to its own min..max
do_clahe            = False               # turn on if illumination is uneven
clahe_clip          = 2.0
clahe_tiles         = (8, 8)
denoise_sigma       = 0.0                 # 0.0 (off) or e.g. 0.8..1.2 to suppress salt/pepper

# Post-filtering & QA
min_size_keep       = 200                 # drop tiny specks directly in CP & post (pixels)
save_overlay        = True

print("project_root:", project_root)
print("img_root    :", img_root)
print("out_root    :", out_root)

# %% [markdown]
# ## 3) Utilities

# %%
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def discover_images(folder: Path, exts) -> list[Path]:
    files = []
    for ext in exts:
        files.extend(sorted(folder.glob(f"*{ext}")))
    # de-dup by stem
    seen, uniq = set(), []
    for p in files:
        if p.stem not in seen:
            uniq.append(p); seen.add(p.stem)
    return uniq

def preprocess_rgb(img: np.ndarray) -> np.ndarray:
    """Per-image intensity rescue for faint signals (RGB or grayscale)."""
    arr = img
    if arr.ndim == 2:
        if do_rescale_intensity:
            arr = exposure.rescale_intensity(arr)
        if denoise_sigma and denoise_sigma > 0:
            arr = filters.gaussian(arr, sigma=denoise_sigma, preserve_range=True).astype(arr.dtype)
        if do_clahe:
            f = exposure.equalize_adapthist(arr, clip_limit=clahe_clip, kernel_size=clahe_tiles)
            arr = util.img_as_ubyte(f) if np.issubdtype(arr.dtype, np.integer) else f
        return arr

    out = arr.copy()
    if do_rescale_intensity:
        for c in range(out.shape[-1]):
            out[..., c] = exposure.rescale_intensity(out[..., c])
    if denoise_sigma and denoise_sigma > 0:
        for c in range(out.shape[-1]):
            out[..., c] = filters.gaussian(out[..., c], sigma=denoise_sigma, preserve_range=True)
        out = out.astype(arr.dtype)
    if do_clahe:
        maxv = np.iinfo(out.dtype).max if np.issubdtype(out.dtype, np.integer) else 1.0
        for c in range(out.shape[-1]):
            f = exposure.equalize_adapthist(out[..., c], clip_limit=clahe_clip, kernel_size=clahe_tiles)
            out[..., c] = (f * maxv).astype(out.dtype)
    return out

def overlay_on_image(img: np.ndarray, labels: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    """RGB overlay for QA."""
    base = np.stack([img]*3, axis=-1) if img.ndim == 2 else img[..., :3]
    b = base.astype(np.float32)
    if b.max() > 0:
        b /= b.max()
    over = color.label2rgb(labels, image=b, bg_label=0, alpha=alpha)
    return (over * 255).astype(np.uint8)

def relabel_and_filter(mask: np.ndarray, min_size: int) -> np.ndarray:
    """Relabel to 1..N and drop small objects (safety after CP's min_size)."""
    lab = measure.label(mask > 0, connectivity=1)
    if min_size > 0:
        lab = morphology.remove_small_objects(lab, min_size=min_size)
        lab = measure.label(lab > 0, connectivity=1)
    return lab.astype(np.int32)

# %% [markdown]
# ## 4) Load Cellpose v4 model
# - v4 path: `models.CellposeModel(pretrained_model="cpsam", device=device, gpu=False)`.

# %%
model = models.CellposeModel(
    gpu=False,                        # do not force CUDA; MPS handled via device
    pretrained_model=pretrained_model,
    device=device
)
print("Loaded:", model.pretrained_model)

# %% [markdown]
# ## 5) Parameter sweep (small sample)
# We search a tiny grid and score each setting to reduce tiny specks while keeping enough objects.
#
# **Heuristic targets (no ground truth):**
# - Lower fraction of very small objects (`min_size_keep`).
# - Reasonable count (not 0; not tens of thousands).
# - Larger median area preferred over noisy tiny specks.
#
# You can adjust the grids below if needed.

# %%
from statistics import median

# small grid — tuned for your data
grid_cellprob = [-2.0, -1.0, 0.0]     # raise → fewer objects
grid_flow     = [0.2, 0.4]            # raise → fewer objects / stricter boundaries
grid_diams    = [30.0, 45.0, 60.0]    # bigger → larger objects; avoids small specks

def score_result(areas, n_labels, tiny_frac):
    # Lower is better. Penalize too few/many objects; prefer fewer tinies, larger areas.
    if n_labels == 0:
        return 1e9
    med = median(areas) if areas else 0
    # heuristic weights
    return (tiny_frac * 3.0) + (abs(np.log10(n_labels + 1) - 2.0) * 1.5) + (1.0 / (med + 1e-6))

def run_once(imgs, cellprob, flow, diam, invert=False, min_size=200):
    masks, _, _ = model.eval(
        imgs,
        diameter=diam,
        batch_size=len(imgs),
        channel_axis=-1,
        invert=invert,
        normalize=False,
        flow_threshold=flow,
        cellprob_threshold=cellprob,
        min_size=min_size,             # v4 supports min_size in eval
    )
    # collect stats on first image only (representative)
    m0 = masks[0].astype(np.int32)
    lab = relabel_and_filter(m0, min_size)
    props = measure.regionprops(lab)
    areas = [p.area for p in props]
    tiny = sum(a < min_size for a in areas)
    tiny_frac = tiny / max(1, len(areas))
    s = score_result(areas, len(areas), tiny_frac)
    return s, lab, len(areas), tiny_frac, (np.median(areas) if areas else 0)

# pick a few sample images
def pick_sample_files(root: Path, n=3):
    fs = discover_images(root, image_exts)
    return fs[:min(n, len(fs))]

# Sweep on yellow
sample_dir = img_root / "yellow"
sample_files = pick_sample_files(sample_dir, n=3)
print("Sweep on:", [p.name for p in sample_files])

# Read + preprocess
sample_raw = [io.imread(p) for p in sample_files]
sample_imgs = [preprocess_rgb(im) for im in sample_raw]

best = None
results = []
for cp in grid_cellprob:
    for fl in grid_flow:
        for dm in grid_diams:
            s, lab, nlab, tinyf, amed = run_once(sample_imgs, cp, fl, dm, invert=False, min_size=min_size_keep)
            results.append(dict(cellprob=cp, flow=fl, diam=dm, score=s, n_labels=nlab, tiny_frac=tinyf, area_med=float(amed)))
            if (best is None) or (s < best["score"]):
                best = dict(cellprob=cp, flow=fl, diam=dm, score=s, n_labels=nlab, tiny_frac=tinyf, area_med=float(amed), lab=lab)

print("Best (heuristic):", {k: best[k] for k in ["cellprob","flow","diam","score","n_labels","tiny_frac","area_med"]})

# Optional: save the best overlay for quick visual QC
ensure_dir(out_root / "yellow" / "sweep_qc")
if sample_imgs:
    ov = overlay_on_image(sample_imgs[0], best["lab"])
    io.imsave(out_root / "yellow" / "sweep_qc" / f"sweep_best_overlay.png", ov, check_contrast=False)

# Save sweep table
pd.DataFrame(results).sort_values("score").to_csv(out_root / "yellow" / "sweep_qc" / "sweep_results.csv", index=False)

# %% [markdown]
# ## 6) Segment ALL yellow images with the chosen settings
# We apply the best `(cellprob, flow, diameter)` found above and save:
# - `*_masks.png` (uint16 labels)
# - `*_overlay.png` (RGB visualization)
# - `segmentation_stats.csv` (objects/areas per image)

# %%
# Use best settings from the sweep
cellprob_threshold = float(best["cellprob"])
flow_threshold     = float(best["flow"])
diameter           = float(best["diam"])

print("Running full set with:",
      "diameter=", diameter,
      "cellprob_threshold=", cellprob_threshold,
      "flow_threshold=", flow_threshold,
      "min_size_keep=", min_size_keep)

stats_rows = []
failed = []
total_images = 0

for ch in channels:                         # ["yellow"]
    in_dir  = img_root / ch
    out_dir = out_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    print(f"{ch}: {len(files)} images -> {out_dir}")

    # simple batching
    def chunked(seq, n): 
        for i in range(0, len(seq), n):
            yield seq[i:i+n]
    batch_size = 4

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Cellpose {ch}"):
        raws = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raws]

        try:
            masks_list, flows, styles = model.eval(
                imgs,
                diameter=diameter,
                batch_size=len(imgs),
                channel_axis=-1,        # RGB inputs
                invert=False,
                normalize=False,
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
                min_size=min_size_keep,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, im, m in zip(group, raws, masks_list):
            lab = relabel_and_filter(m.astype(np.int32), min_size_keep)

            # Save mask
            mpath = out_dir / f"{p.stem}_masks.png"
            io.imsave(mpath, lab.astype(np.uint16), check_contrast=False)

            # Save overlay
            if save_overlay:
                ov = overlay_on_image(preprocess_rgb(im), lab)
                io.imsave(out_dir / f"{p.stem}_overlay.png", ov, check_contrast=False)

            # Stats
            props = measure.regionprops(lab)
            areas = [pr.area for pr in props]
            stats_rows.append({
                "file": p.name,
                "n_labels": len(props),
                "area_median": float(np.median(areas)) if areas else 0.0,
                "area_mean": float(np.mean(areas)) if areas else 0.0,
                "area_min": int(min(areas)) if areas else 0,
                "area_max": int(max(areas)) if areas else 0,
                "tiny_frac(<min_size)": float(sum(a < min_size_keep for a in areas) / max(1, len(areas)))
            })
        total_images += len(group)

print("Done. Images segmented:", total_images, "| failures:", len(failed))
ensure_dir(out_root / "yellow")
pd.DataFrame(stats_rows).to_csv(out_root / "yellow" / "segmentation_stats.csv", index=False)
print("Stats CSV:", out_root / "yellow" / "segmentation_stats.csv")


python: 3.10.18
platform: macOS-15.5-arm64-arm-64bit
torch: 2.7.1
MPS available: True


Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


Using device: mps
project_root: /Users/ashi/github/cm4ai_codefest2025
img_root    : /Users/ashi/github/cm4ai_codefest2025/data
out_root    : /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2
Loaded: /Users/ashi/.cellpose/models/cpsam
Sweep on: ['B2AI_1_Paclitaxel_A1_R2_z01_yellow.jpg', 'B2AI_1_Paclitaxel_A1_R5_z01_yellow.jpg', 'B2AI_1_Paclitaxel_A1_R6_z01_yellow.jpg']


Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is de

Best (heuristic): {'cellprob': -2.0, 'flow': 0.2, 'diam': 30.0, 'score': 1000000000.0, 'n_labels': 0, 'tiny_frac': 0.0, 'area_med': 0.0}
Running full set with: diameter= 30.0 cellprob_threshold= -2.0 flow_threshold= 0.2 min_size_keep= 200
yellow: 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


Cellpose yellow:  67%|██████▋   | 2/3 [02:44<01:21, 81.30s/it]/var/folders/5j/mk3rz4214w99nf1062dvwx3m0000gn/T/ipykernel_14964/2944410911.py:138: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?
  lab = morphology.remove_small_objects(lab, min_size=min_size)
Cellpose yellow: 100%|██████████| 3/3 [03:26<00:00, 68.71s/it]

Done. Images segmented: 10 | failures: 0
Stats CSV: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/segmentation_stats.csv
